# 00: Colab setup check (environment only, no inference)

This notebook validates the Colab runtime before `01_colab_inference.ipynb`. It checks the host and GPU, mounts
Drive, clones the runtime repo, installs the runtime requirements, detects the accelerator and confirms that the
frozen bundle and competition files are on Drive. Nothing is copied, hashed, tested or inferred.

Use **Runtime → Change runtime type → GPU** (T4 or better). A CPU runtime also works: inference then runs on the
validated CPU backend.

In [ ]:
# CELL 1 -- runtime information (never fails on a CPU runtime)
import os, platform, sys
print('python :', sys.version.split()[0])
print('os     :', platform.platform())
print('cpu    :', platform.processor() or platform.machine(), '| cores:', os.cpu_count())
try:
    import psutil
    print(f'ram    : {psutil.virtual_memory().total / 1024 ** 3:.1f} GB | disk free /content: {psutil.disk_usage("/content").free / 1024 ** 3:.1f} GB')
except ImportError:
    print('ram    : psutil not installed yet')
!nvidia-smi || echo "nvidia-smi unavailable -> CPU runtime (inference will use the validated CPU backend)"

In [ ]:
# CELL 2 -- mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3 -- clone / update the GitHub runtime repo. A private repo uses the Colab Secret GITHUB_TOKEN (key icon ->
# Secrets). The token is never hard-coded, printed, written to .git/config or placed in os.environ.
import base64, subprocess
from pathlib import Path

GITHUB_USER = 'YOUR_GITHUB_USER'          # <- your GitHub account
REPO_NAME = 'enveda-casmi-runtime'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content') / REPO_NAME
assert GITHUB_USER != 'YOUR_GITHUB_USER', 'set GITHUB_USER'


def _github_token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return None                                 # public repo, or the secret is not granted to this notebook


def _git(args):
    token = _github_token()
    auth = []
    if token:
        basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        auth = ['-c', f'http.extraHeader=Authorization: Basic {basic}']   # lives only for this one git call
    r = subprocess.run(['git', *auth, *args], capture_output=True, text=True)
    del token, auth
    if r.returncode != 0:
        raise RuntimeError(f'git {args[0]} failed (exit {r.returncode}):\n{r.stderr[-2000:]}')
    return r.stdout


if (REPO_DIR / '.git').exists():
    _git(['-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    _git(['clone', '--depth', '1', '--branch', REPO_BRANCH, f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git', str(REPO_DIR)])
print('runtime repo at', REPO_DIR, '@', subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
# CELL 4 -- runtime requirements only (pip skips everything Colab already satisfies)
import sys
from importlib.metadata import PackageNotFoundError, version
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
r.check_returncode()
for pkg in ('numpy', 'pandas', 'pyarrow', 'lightgbm', 'numba', 'psutil', 'rdkit', 'pytest'):
    try:
        print(f'{pkg:10s} {version(pkg)}')
    except PackageNotFoundError:
        print(f'{pkg:10s} NOT INSTALLED')
assert version('lightgbm') == '4.7.0', 'lightgbm must be 4.7.0 (frozen boosters): Runtime -> Restart session, then re-run from CELL 3'

In [ ]:
# CELL 5 -- accelerator detection (the reusable script; hardware only -- backend choice happens at inference time)
r = subprocess.run([sys.executable, str(REPO_DIR / 'scripts' / 'detect_accelerator.py')], capture_output=True, text=True)
print(r.stdout, r.stderr[-1000:])

In [ ]:
# CELL 6 -- are the frozen bundle and the competition files on Drive? (presence + identity only; no hashing, no inference)
import json
DRIVE_ROOT = Path('/content/drive/MyDrive/EnvedaCASMI')
checks = {'bundle/manifest.json': DRIVE_ROOT / 'bundle' / 'manifest.json', 'bundle/config.json': DRIVE_ROOT / 'bundle' / 'config.json',
          'competition/test.parquet': DRIVE_ROOT / 'competition' / 'test.parquet',
          'competition/sample_submission.csv': DRIVE_ROOT / 'competition' / 'sample_submission.csv'}
for k, p in checks.items():
    print(f"{'OK ' if p.exists() else 'MISSING'} {k}")
assert all(p.exists() for p in checks.values()), 'copy the ENTIRE bundle/ + competition files to MyDrive/EnvedaCASMI (see COLAB_RUN_GUIDE.md)'
cfg = json.loads(checks['bundle/config.json'].read_text(encoding='utf-8'))
man = json.loads(checks['bundle/manifest.json'].read_text(encoding='utf-8'))
missing = [f for f in man['files'] if not (DRIVE_ROOT / 'bundle' / f).exists()]
print({k: cfg.get(k) for k in ('bundle_version', 'model_id', 'aggregator_name', 'calibration_temperature', 'CONFIG_HASH')},
      '| manifest bundle_version:', man.get('bundle_version'), f"| {len(man['files']) - len(missing)}/{len(man['files'])} manifest files present on Drive")
assert not missing, f'incomplete bundle on Drive (e.g. {missing[:5]}) -- copy the whole directory again'
print('SETUP OK -- continue with notebooks/01_colab_inference.ipynb')